In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from model import ImprovedCNN

In [ ]:
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"device:{device}")


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),
                         (0.5,0.5,0.5))
])


test_dataset = datasets.CIFAR10(
    "./data",
    train=False,
    download=True,
    transform=transform
)


loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
model = ImprovedCNN().to(device)

model.load_state_dict(
    torch.load(
        "cifar10_improved_cnn.pth",
        map_location=device
    )
)

model.eval()

predictions = []
labels = []


with torch.no_grad():

    for images, targets in loader:

        images = images.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        predictions.extend(
            predicted.cpu().numpy()
        )

        labels.extend(
            targets.numpy()
        )

In [ ]:
accuracy = sum(
    p == l for p,l in zip(predictions, labels)
) / len(labels)


print(f"Accuracy: {accuracy*100:.2f}%")

print(
    classification_report(
        labels,
        predictions,
        target_names=classes
    )
)


cm = confusion_matrix(
    labels,
    predictions
)


plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Improved CNN Confusion Matrix")

plt.tight_layout()

plt.savefig(
    "results/confusion_matrix.png",
    dpi=300
)